# Working with Dates and Times in Python
### INFO 648 - Business Data Analytics
**Presenters:** Haojing Shen & Sathvik Nagaraj

**Date:** October 29, 2026

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, datetime

# Part 1: Dates and Times Basics

## What Are Dates, Times, and Datetimes?

| Concept | Meaning | Example | Python type |
|---|---|---|---|
| **Date** | A calendar day | April 28, 2026 | `date` |
| **Time** | A time of day | 6:45 PM | `time` |
| **Datetime** | A date and a time together (one exact moment) | April 28, 2026, 6:45 PM | `datetime` |

A **timestamp** is a datetime recorded for an event, such as the moment an order was placed.

## Why Do Dates Matter in Analytics?

Sales, orders, visits, and payments all come with dates. Dates let analysts answer questions like *"Which month sold the most?"* and *"Are sales growing?"*

If dates are handled incorrectly, the report can still look normal and be **wrong**. That is what we will practice avoiding today.


## Creating Dates and Datetimes

Use `date(year, month, day)` and `datetime(year, month, day, hour, minute)`.

In [ ]:
order_date = date(2026, 4, 28)                  # a date
order_moment = datetime(2026, 4, 28, 18, 45)    # a datetime (6:45 PM, 24-hour clock)

print(order_date)
print(order_moment)
print(type(order_moment).__name__)

## Extracting Year, Month, and Day

In [ ]:
print("Year :", order_moment.year)
print("Month:", order_moment.month)
print("Day  :", order_moment.day)

## Formatting Dates with `strftime()`

`strftime()` turns a datetime **into text** using a pattern.

| Code | Meaning | Example |
|---|---|---|
| `%Y` | 4-digit year | 2026 |
| `%m` | Month number | 04 |
| `%b` / `%B` | Short / full month name | Apr / April |
| `%d` | Day of month | 28 |
| `%A` | Weekday name | Tuesday |

In [ ]:
print(order_moment.strftime("%Y-%m-%d"))     # 2026-04-28
print(order_moment.strftime("%b %d, %Y"))    # Apr 28, 2026
print(order_moment.strftime("%A, %B %d"))    # Tuesday, April 28

## Converting Text into Dates with `strptime()`

`strptime()` does the reverse: it reads **text** and returns a datetime. You must describe the text's pattern exactly.

In [ ]:
text = "Apr 28, 2026"
converted = datetime.strptime(text, "%b %d, %Y")   # pattern matches the text

print(converted, "->", type(converted).__name__)
print("Month number:", converted.month)

## A String Is Not a Date

The text `"Apr 28, 2026"` only *looks* like a date. Python sees it as characters, so it cannot read the month from it.

In [ ]:
print(type(text).__name__, "vs", type(converted).__name__)
print("hasattr(text, 'month')     :", hasattr(text, "month"))
print("hasattr(converted, 'month'):", hasattr(converted, "month"))

# Part 2: Working with Dates in pandas

## Sample Data: Café Sales

pandas stores data in tables called DataFrames. Below is a small set of café transactions. The dates are **text** (like `"Jan 05, 2026"`), the way they often arrive from a spreadsheet or CSV file.

In [ ]:
records = [
    ("Mar 14, 2026", "K103", "Merchandise", 48.00),
    ("Jan 05, 2026", "K101", "Beverages", 12.50),
    ("Feb 20, 2026", "K104", "Bakery", 18.75),
    ("Jan 19, 2026", "K102", "Merchandise", 35.00),
    ("Apr 02, 2026", "K101", "Beverages", 14.25),
    ("Mar 03, 2026", "K105", "Bakery", 22.00),
    ("Feb 07, 2026", "K103", "Beverages", 16.00),
    ("Apr 17, 2026", "K104", "Merchandise", 52.50),
    ("Jan 28, 2026", "K105", "Bakery", 9.75),
    ("Mar 25, 2026", "K102", "Beverages", 21.00),
    ("Feb 14, 2026", "K101", "Bakery", 27.50),
    ("Apr 28, 2026", "K103", "Beverages", 19.25),
    ("Mar 09, 2026", "K104", "Beverages", 13.00),
    ("Jan 11, 2026", "K103", "Bakery", 11.25),
    ("Feb 26, 2026", "K105", "Merchandise", 44.00),
    ("Apr 10, 2026", "K102", "Bakery", 24.50),
]

sales_raw = pd.DataFrame(
    records,
    columns=["order_date", "customer_id", "product_category", "sales_amount"]
)

sales_raw.head()

In [ ]:
sales_raw.dtypes      # order_date shows as 'object' or 'str': plain text, not dates

## Converting with `pd.to_datetime()`

`pd.to_datetime()` converts a whole column of text into real dates.

- `format="%b %d, %Y"` tells pandas the text's pattern (same codes as `strptime`).
- `errors="coerce"` turns any invalid date into `NaT` (Not a Time = a missing date) instead of stopping with an error.

In [ ]:
sales = sales_raw.copy()                                   # keep the original text version safe
sales["order_date"] = pd.to_datetime(sales["order_date"], format="%b %d, %Y")

sales.dtypes          # order_date is now datetime64

### How `errors="coerce"` Behaves *(Optional)*

In [ ]:
test_dates = pd.Series(["Jan 05, 2026", "Feb 30, 2026", "not a date"])   # Feb 30 does not exist

pd.to_datetime(test_dates, format="%b %d, %Y", errors="coerce")

Valid text becomes a date; the impossible date and the nonsense text both become `NaT`. Always count your `NaT` values afterward with `.isna().sum()`.

## Extracting Date Components with `.dt`

In [ ]:
sales["year"] = sales["order_date"].dt.year
sales["month"] = sales["order_date"].dt.month
sales["day"] = sales["order_date"].dt.day

sales[["order_date", "year", "month", "day"]].head()

## Sorting Records by Date

In [ ]:
sales = sales.sort_values("order_date").reset_index(drop=True)   # oldest first

sales[["order_date", "customer_id", "sales_amount"]].head(6)

## Filtering Records by Date

Keep only the rows between two dates, using `&` to combine two conditions.

In [ ]:
start = pd.Timestamp("2026-02-01")
end = pd.Timestamp("2026-02-28")

february = sales[(sales["order_date"] >= start) & (sales["order_date"] <= end)]

print("February orders:", len(february))
print(f"February sales: ${february['sales_amount'].sum():,.2f}")
february[["order_date", "customer_id", "sales_amount"]]

## Grouping Sales by Month

`.dt.to_period("M")` labels each row with its year and month (like `2026-03`). Then `groupby()` collects the rows of each month and `.sum()` adds up their sales.

In [ ]:
sales["month_label"] = sales["order_date"].dt.to_period("M")

monthly_sales = sales.groupby("month_label")["sales_amount"].sum()

monthly_sales

> 🎤 *Presenter note: groupby() can also group strings. It’s the preceding .dt.to_period("M") that requires a datetime type.*

# Part 3: Business Example: How Are Sales Changing Over Time?

As the café's analyst, we want to know the **best month** and **how each month compares with the one before it**. We use the same `sales` table.

In [ ]:
best_month = monthly_sales.idxmax()           # the month with the highest total
best_total = monthly_sales.max()
share = best_total / monthly_sales.sum() * 100

print(f"Best month: {best_month.strftime('%B %Y')} with ${best_total:,.2f} in sales")
print(f"That is {share:.1f}% of all sales in the data.")

In [ ]:
change = monthly_sales.pct_change() * 100     # percent change from the previous month

for month, value in change.iloc[1:].items():  # skip the first month (nothing before it)
    direction = "up" if value > 0 else "down"
    print(f"{month.strftime('%B %Y')}: sales {direction} {abs(value):.1f}% from the previous month")

### Visualizing the Trend *(Optional)*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(monthly_sales.index.strftime("%b %Y"), monthly_sales.values, color="#4C78A8")
ax.set_title("Café Sales by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Total sales ($)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### What This Means

The printed results above show which month was strongest and whether sales are rising or falling month to month. A business can use this to plan staffing, inventory, and promotions around busy and slow periods.

*This is a small sample (16 orders), so it shows the method, not real business conclusions.*

# Part 4: Common Mistakes

## Mistake 1: Sorting Dates as Text

If the column is still text, Python sorts **letters**, not calendar dates.

In [ ]:
# Incorrect Example: sorting the text column

wrong_sorted = sales_raw.sort_values("order_date")

print("Order of dates:", wrong_sorted["order_date"].tolist()[:6], "...")
print("'Latest' sale according to the text:", sales_raw["order_date"].max())

In [ ]:
# Correct Example: convert first, then sort

print("Order of dates:", [d.strftime("%b %d") for d in sales["order_date"][:6]], "...")
print("Latest sale:", sales["order_date"].max().strftime("%b %d, %Y"))

**Why it fails:** alphabetically, April comes before February, January, and March, so the timeline is scrambled and the "latest" sale is wrong. No error message appears, which makes this mistake easy to miss.

## Mistake 2: Guessing the Date Format

What does `04/05/2026` mean: **April 5** or **May 4**? Numbers alone cannot tell you.

In [ ]:
# Incorrect Example: reading day-first dates as month-first (or the reverse)

small = pd.DataFrame({"date_text": ["01/02/2026", "02/03/2026", "03/04/2026", "04/05/2026"],
                      "sales": [100, 200, 300, 400]})

wrong = pd.to_datetime(small["date_text"], format="%d/%m/%Y")   # assumed DAY first

# Suppose the file is actually MONTH first (US style):
right = pd.to_datetime(small["date_text"], format="%m/%d/%Y")

print("Text      Read as day-first   Read as month-first")
for t, w, r in zip(small["date_text"], wrong, right):
    print(f"{t}  {w.strftime('%b %d'):>17}   {r.strftime('%b %d'):>19}")

In [ ]:
# Correct Example: confirm the format with the data's source, then state it explicitly

dates = pd.to_datetime(small["date_text"], format="%m/%d/%Y")   # source confirmed: MONTH/DAY/YEAR

print(dates.dt.month_name().tolist())

**Why it fails:** every row here is a valid date either way, so pandas gives no warning. But each sale lands in a **different month**, which changes the monthly totals. Always confirm the format and write it in `format=`. Prefer the unambiguous `YYYY-MM-DD` style when you can.

## Mistake 3: Forgetting That an End Date Means Midnight

When a column has **times** as well as dates, `<= "2026-03-31"` means "up to 12:00 AM on March 31," so transactions later that day are dropped.

In [ ]:
checkouts = pd.DataFrame({
    "timestamp": pd.to_datetime(["2026-03-30 10:15", "2026-03-31 09:05",
                                 "2026-03-31 13:40", "2026-03-31 18:55", "2026-04-01 08:30"]),
    "amount": [10.00, 15.00, 20.00, 25.00, 30.00],
})

# Incorrect Example: "through March 31"
wrong_march = checkouts[checkouts["timestamp"] <= "2026-03-31"]
print("Orders kept:", len(wrong_march), "| March total: $", wrong_march["amount"].sum())

In [ ]:
# Correct Example: everything BEFORE the start of April 1

march = checkouts[(checkouts["timestamp"] >= "2026-03-01") & (checkouts["timestamp"] < "2026-04-01")]
print("Orders kept:", len(march), "| March total: $", march["amount"].sum())

**Why it fails:** the three orders placed later on March 31 were left out. In month-end reports, the last day is often a busy one. Rule of thumb: *start at the first moment of the period, and end before the first moment of the next period.*

# Summary

- **Dates matter:** correct date handling decides whether monthly totals, trends, and comparisons can be trusted.
- **Strings vs. datetimes:** text that looks like a date cannot be sorted, filtered, or grouped by calendar until it is converted with `strptime()` or `pd.to_datetime()`.
- **pandas makes it easy:** `pd.to_datetime()` converts a column, `.dt` extracts parts, and sorting, filtering, and `groupby()` then work on the calendar.
- **Avoid common errors:** sorting dates as text, guessing the date format, and mishandling end dates when timestamps include a time.
- **In real analytics:** sales reports, trend tracking, and planning all start with trustworthy dates.

**Thank you! Questions?**